# Product Search Relevance and Optimization with RedisVL

<a href="https://colab.research.google.com/github/redis-developer/search-workshop/blob/colab-migration/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Workshop Goal

You are improving search for an e-commerce catalog. The goal is not merely to return products; it is to build a retrieval workflow that you can explain, measure, and improve with evidence.

The workshop follows one search-quality loop:

`Prepare data -> Represent products -> Build an index -> Retrieve candidates -> Evaluate -> Decide`

### 60-Minute Route

| Time | Focus | Decision you can make afterward |
|---:|---|---|
| 0-8 min | WANDS data and relevance judgments | What constitutes evidence for search quality? |
| 8-16 min | Embeddings | Which model sourcing approach fits the workload? |
| 16-25 min | Vector index types and loading | Why is `FLAT` the right live baseline, and when would `HNSW` or `SVS-VAMANA` replace it? |
| 25-37 min | Retrieval patterns | When should text, vector, filters, or hybrid retrieval be used? |
| 37-40 min | Production index benchmark | How should approximate indexes be compared fairly? |
| 40-55 min | Relevance optimization | Which retrieval method wins on judged queries? |
| 55-60 min | Recommendation | What should the next production experiment be? |

### Outcomes

By the end, you will be able to:

- Prepare the full judged product-search corpus from the [WANDS dataset](https://github.com/wayfair/WANDS).
- Turn product metadata into `search_text` and embeddings, and explain hosted, open-weight, and fine-tuned model choices.
- Explain how `FLAT`, `HNSW`, and `SVS-VAMANA` work, then build the exact `FLAT` baseline with RedisVL.
- Compare vector search, tag- and numeric-filtered vector search, and `FT.HYBRID`.
- Score retrieval methods with nDCG@10, Recall@25, Precision@25, and query time.
- Turn the evidence into a practical next experiment.

> **Workshop contract:** Every run uses all 42,994 products, all 480 queries, and all available relevance judgments.


## 0. Bootstrap the Workshop

The setup cells make a fresh Google Colab runtime self-contained. They clone the complete workshop repository from GitHub, install its declared Python dependencies, and run a local Redis 8.6 instance with Redis Search support.

Outside Colab, the repository clone and Redis installation steps are skipped so the same notebook continues to work with the local `uv` and Docker Compose workflow.


### 0.1 Clone the Supporting Repository

Colab opens this notebook from GitHub without checking out its sibling files. This cell clones the `colab-migration` branch into `/content/search-workshop`, changes the working directory, and verifies that the configuration, code, and project requirements are present. Rerunning it fast-forwards the existing branch clone.


In [ ]:
from pathlib import Path
import os
import subprocess

try:
    import google.colab  # type: ignore[import-not-found]
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

REPO_URL = 'https://github.com/redis-developer/search-workshop.git'
REPO_REF = 'colab-migration'
REPO_DIR = Path('/content/search-workshop') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if (REPO_DIR / '.git').is_dir():
        active_ref = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'branch', '--show-current'],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
        if active_ref != REPO_REF:
            raise RuntimeError(
                f'{REPO_DIR} is on {active_ref!r}, not {REPO_REF!r}. '
                'Restart the Colab runtime before running this branch notebook.'
            )
        subprocess.run(
            ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_REF],
            check=True,
        )
    elif REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository. Start a fresh runtime or move that directory.')
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )

os.chdir(REPO_DIR)

required_artifacts = [
    'pyproject.toml',
    '.env.example',
    'scripts/prep_wands.py',
    'scripts/setup_colab_redis.py',
    'scripts/workshop_search_study.py',
    'scripts/validate.py',
]
missing_artifacts = [name for name in required_artifacts if not (REPO_DIR / name).is_file()]
if missing_artifacts:
    raise FileNotFoundError(f'Missing cloned workshop artifacts: {missing_artifacts}')

commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f'Workshop root: {REPO_DIR}')
print(f'Git commit: {commit}')
print(f'Cloned supporting artifacts: {", ".join(required_artifacts)}')

### 0.2 Install Python Dependencies

`pyproject.toml` is the single dependency source for Colab and local development. Colab already supplies the notebook server, so JupyterLab remains a local-development dependency rather than part of this runtime install.


In [ ]:
if IN_COLAB:
    %pip install -q .
else:
    print('Not running in Colab; use `uv sync` to manage local dependencies.')

### 0.3 Start Redis 8.6 in Colab

Redis 8.6 uses the unified Redis Open Source distribution. The cloned setup script installs the newest available `8.6.*` patch from the official Redis APT repository, loads both Search and RedisJSON when needed, and verifies `PING`, the server version, `FT.HYBRID`, and `JSON.GET` before continuing.

The APT pin is intentional: installing the unversioned latest Redis package could move this workshop to a newer minor release. Outside Colab, this cell leaves the Redis process unchanged.


In [ ]:
if IN_COLAB:
    from scripts.setup_colab_redis import COLAB_REDIS_URL, setup_colab_redis

    redis_state = setup_colab_redis()
    print(f"Redis {redis_state['version']} is ready at {COLAB_REDIS_URL}.")
    print(f"Loaded modules: {', '.join(sorted(redis_state['modules']))}")
    print(f"Verified commands: {', '.join(sorted(redis_state['commands']))}")
else:
    print('Not running in Colab; use Docker Compose or REDIS_URL for Redis 8.6+.')

## 1. Set Up the Product Search Data

The [Wayfair Annotation Dataset (WANDS)](https://github.com/wayfair/WANDS) gives us the three artifacts required for an offline relevance experiment. Keep their roles separate:

| Artifact | What it contains | Why it matters |
|---|---|---|
| Product corpus | Product names, categories, descriptions, features, and business attributes | The documents Redis will search. |
| Queries | Shopper search phrases | The inputs every retrieval method must answer. |
| Relevance judgments (`qrels`) | Human labels for query/product pairs | The evidence used to score and compare rankings. |

The prep script maps WANDS labels to graded scores: `Exact = 2`, `Partial = 1`, and `Irrelevant = 0`. Graded labels matter because ranking an exact match above a partial match should receive more credit.

> **Qrels count:** The WANDS label file contains 233,448 rows and 231,873 unique query-product pairs. Because qrels require one grade per pair, preparation keeps the highest observed grade when duplicate rows disagree.

A WANDS query can have many relevant products. That is realistic for e-commerce, but it changes how recall should be interpreted: a top-25 result list cannot recover hundreds or thousands of positives. The scorecard defines that natural ceiling explicitly before comparing methods. For background on the dataset and its annotation process, see Wayfair's [WANDS overview](https://www.aboutwayfair.com/careers/tech-blog/wayfair-releases-wands-the-largest-and-richest-publicly-available-dataset-for-e-commerce-product-search-relevance).


### 1.1 Workshop Environment Settings

These values control the Redis connection, embedding model and throughput, and run-scoped Redis names. The defaults come from `.env.example`, and every Redis client is built from `REDIS_URL`.


In [ ]:
from pathlib import Path
import os
import re

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
load_dotenv(ROOT / '.env')

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
REDIS_URL = os.getenv('REDIS_URL', 'redis://localhost:6379')
REDIS_LOAD_BATCH_SIZE = int(os.getenv('REDIS_LOAD_BATCH_SIZE', '25'))
EMBEDDING_CHUNK_SIZE = int(os.getenv('EMBEDDING_CHUNK_SIZE', '1024'))
EMBEDDING_BATCH_SIZE = int(os.getenv('EMBEDDING_BATCH_SIZE', '64'))
HF_MODEL = os.getenv('HF_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')
WORKSHOP_RUN_ID = re.sub(r'[^A-Za-z0-9_]', '_', os.getenv('WORKSHOP_RUN_ID', 'local')).strip('_') or 'local'

print('Workshop configuration')
print('- Redis URL: loaded from environment')
print('- Dataset: full WANDS corpus and judged query set')
print(f'- Embedding model: {HF_MODEL}')
print(f'- Workshop run ID: {WORKSHOP_RUN_ID}')
print('- Search study: all loaded queries')
print(f'- Redis load batch size: {REDIS_LOAD_BATCH_SIZE:,}')
print(f'- Embedding chunk size: {EMBEDDING_CHUNK_SIZE:,}')
print(f'- Embedding batch size: {EMBEDDING_BATCH_SIZE:,}')

### 1.2 Operational Notes

- **Redis:** The bootstrap starts and verifies Redis in Colab. Locally, start Docker Compose or provide `REDIS_URL` before the RedisVL sections.
- **Run isolation:** `WORKSHOP_RUN_ID` scopes the index name, key prefix, and embedding caches. Use `local` on one laptop; use initials or a seat number on shared Redis.
- **Repeatability:** The load cell uses `overwrite=True, drop=True` to recreate only the run-scoped workshop index. Never reuse these names for a production index.
- **Colab lifetime:** The repository clone, generated data, embedding cache, and local Redis process disappear when the runtime resets.

> **Full-data expectation:** Initial embedding and loading takes longer than a cache-warm rerun because every product is processed. Evaluation always uses every loaded query and judgment.


### 1.3 Prepare the WANDS Dataset

The prep script downloads WANDS, validates the raw files, and writes the complete product corpus, query set, and relevance judgments. The notebook has one data path: full WANDS.


In [ ]:
from scripts.prep_wands import prepare_wands

manifest = prepare_wands(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
)

source = manifest['source']

print(f"Dataset: {manifest['dataset']}")
print(f"Source:  {source['homepage']}")
print(source['note'])
print()
print('Raw WANDS counts')
print(f"- Products:  {source['raw_counts']['products']:,}")
print(f"- Queries:   {source['raw_counts']['queries']:,}")
print(f"- Judgments: {source['raw_counts']['judgments']:,}")
print()

selected_files = manifest['files']['full']

print('Prepared full WANDS counts')
print(f"- Products:  {int(selected_files['products']):,}")
print(f"- Queries:   {int(selected_files['queries_count']):,}")
print(f"- Judgments: {int(selected_files['qrels_count']):,}")


### 1.4 Load Corpus, Queries, and Judgments

Now load the prepared files into memory. Redis is not involved yet; this is just local data setup.


In [ ]:
import json

import pandas as pd

pd.set_option('display.max_colwidth', 90)


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

corpus = read_json(selected_files['corpus'])
queries = read_json(selected_files['queries'])
qrels = read_json(selected_files['qrels'])

corpus_df = pd.DataFrame(corpus.values())
corpus_by_id = corpus

print(f"Loaded {len(corpus_df):,} products, {len(queries):,} queries, and {sum(len(items) for items in qrels.values()):,} judgments.")
print(f"Search study will use all {len(queries):,} queries.")

### 1.5 Choose One Query to Follow

We will use one readable query through the query-pattern section. That makes it easier to compare how each retrieval method behaves.


In [ ]:
from collections import Counter


def relevant_product_ids(qid):
    return [pid for pid, score in qrels[qid].items() if score > 0 and pid in corpus_by_id]


def teaching_filter_class(qid):
    classes = [corpus_by_id[pid]['product_class'] for pid in relevant_product_ids(qid)]
    nonblank_classes = [value for value in classes if value]
    return Counter(nonblank_classes or classes).most_common(1)[0][0]


def choose_demo_query():
    for preferred in ['writing desk', 'unique coffee tables', 'card table']:
        for qid, text in queries.items():
            if text == preferred and relevant_product_ids(qid):
                return qid
    return next(qid for qid in queries if relevant_product_ids(qid))


demo_qid = choose_demo_query()
demo_query = queries[demo_qid]

print(f"Demo query: {demo_query!r}  (query_id={demo_qid})")

Preview a few product records and the exact `search_text` that will be embedded and indexed.


In [ ]:
preview_df = corpus_df[['product_id', 'product_name', 'product_class', 'search_text']].head(5).copy()
preview_df['search_text'] = preview_df['search_text'].str.slice(0, 180) + '...'
display(preview_df)

## 2. Generate and Cache Embeddings

Each product is represented by one embedding generated from `search_text`, which combines its name, class, category hierarchy, description, and features. Before choosing a vectorizer, separate the representation itself from how its model is sourced and operated.

RedisVL's `EmbeddingsCache` avoids repeated model work during notebook reruns. `WORKSHOP_RUN_ID` isolates participants on a shared database, while deterministic text/model keys make cache hits repeatable. The workshop uses `ttl=None`; production systems should define cache ownership and retention separately from the canonical embeddings stored with product data.


### 2.1 What Embeddings Encode and How to Choose a Model

A text embedding model maps variable-length text to a fixed-length numeric vector. In this lab:

- The same bi-encoder independently encodes each query and product `search_text`.
- Redis compares the vectors with cosine distance; smaller distance means greater directional similarity.
- Product vectors are computed before shopper queries arrive, which makes them practical for candidate retrieval.

> **Model contract:** Query and product vectors must share a compatible embedding space, and the index dimension must match the model output. Changing the model, dimensions, or required query/document prompts normally requires re-embedding the corpus and rebuilding the vector index.

Hosted, open-weight, and fine-tuned are best understood as **model sourcing and adaptation choices**, not different vector index types:

| Choice | Prefer it when | Main tradeoffs |
|---|---|---|
| Hosted embedding API | You want a managed service, quick adoption, and no model-serving infrastructure. | Per-request cost, network latency, rate limits, provider dependency, and data-governance review. |
| Open-weight model you run | You need local execution, privacy, model control, or predictable cost at sustained volume. | You own model selection, serving capacity, upgrades, and observability. |
| Fine-tuned embedding model | A measured baseline shows repeated domain-specific errors and you have reliable query-positive-negative training examples. | Training and evaluation work, overfitting risk, model versioning, and a full corpus re-embed. |

> **Workshop choice:** [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) is small, local, repeatable, and requires no API key. It is a good teaching model, not an automatic production recommendation. RedisVL's [hosted and local vectorizers](https://redis.io/docs/latest/develop/ai/redisvl/concepts/utilities/) let production candidates use the same retrieval workflow.

Before fine-tuning, measure simpler changes:

- Improve `search_text` composition.
- Test a stronger off-the-shelf embedding model.
- Add hybrid retrieval or reranking.

Fine-tune only when the baseline shows repeatable domain errors and you have reliable query-positive-negative examples. Keep training and evaluation queries separate. The [Sentence Transformers training overview](https://sbert.net/docs/sentence_transformer/training_overview.html) covers dataset formats, loss functions, hard negatives, and retrieval evaluators.


### 2.2 Connect to Redis

Redis is required for the rest of the lab. This ping catches connection problems before we spend time embedding products.

In [ ]:
from redis import Redis

redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()

### 2.3 Build the Vectorizer

The vectorizer turns product and query text into compatible vectors. Its reported dimensions become part of the index schema contract in Section 3, and the cache avoids recomputing identical inputs across notebook reruns.

In [ ]:
import warnings

warnings.filterwarnings('ignore', message='IProgress not found.*')

from redisvl.extensions.cache.embeddings import EmbeddingsCache
from redisvl.utils.vectorize import HFTextVectorizer

HF_CACHE_NAME = f'wands_hf_embeddings_{WORKSHOP_RUN_ID}'

embedding_cache = EmbeddingsCache(
    name=HF_CACHE_NAME,
    redis_url=REDIS_URL,
    ttl=None,
)

vectorizer = HFTextVectorizer(
    model=HF_MODEL,
    dtype='float32',
    cache=embedding_cache,
)

EMBEDDING_DIMS = vectorizer.dims
print(f'Using {HF_MODEL} with {EMBEDDING_DIMS} dimensions.')
print(f'Embedding cache: {HF_CACHE_NAME}')

### 2.4 Embed the Product Corpus

This adds one vector per product. Full WANDS runs can take several minutes, so the helper logs progress by outer chunk while the vectorizer handles model batches internally.


In [ ]:
def embed_texts_with_progress(texts, chunk_size, batch_size):
    embeddings = []
    total = len(texts)

    for start in range(0, total, chunk_size):
        chunk = texts[start:start + chunk_size]
        embeddings.extend(vectorizer.embed_many(
            chunk,
            batch_size=batch_size,
            as_buffer=True,
            normalize_embeddings=True,
        ))
        loaded = min(start + len(chunk), total)
        print(f'Embedded {loaded:,}/{total:,} products...')

    return embeddings


corpus_df['embedding'] = embed_texts_with_progress(
    corpus_df['search_text'].tolist(),
    chunk_size=EMBEDDING_CHUNK_SIZE,
    batch_size=EMBEDDING_BATCH_SIZE,
)

print(f"Embedded {len(corpus_df):,} products.")

> **Checkpoint:** Every product now has one normalized vector. Queries must use the same compatible model configuration, and the Redis vector field must use the dimensions printed above.


## 3. Create the RedisVL Search Index

The schema is the contract between product data and retrieval behavior:

- Text fields support lexical search.
- Tag and numeric fields support filters.
- The vector field supports semantic search.


### 3.1 Choose a Vector Index Type

The vector index type determines how Redis finds nearest neighbors. It is separate from the embedding model, distance metric, filters, and hybrid ranking strategy. Keeping `search_text` and `embedding` in one search index lets a query combine those independent signals.

| Vector index type | What happens under the hood | Important controls | Main tradeoff |
|---|---|---|---|
| `FLAT` | Redis computes the query distance to every indexed vector, then returns the exact top-k. Work grows with the number and dimension of vectors. | Datatype, dimensions, and distance metric. | Exact and easy to debug, but query latency grows linearly with the corpus. |
| `HNSW` | Redis traverses a multi-layer proximity graph. Sparse upper layers make large jumps; denser lower layers refine the candidate set. | `M` controls graph degree, `EF_CONSTRUCTION` controls build quality, and `EF_RUNTIME` controls the query recall/latency tradeoff. | Fast approximate search at scale, with added graph memory, build time, and tuning. |
| `SVS-VAMANA` | Redis traverses a bounded-degree proximity graph using a candidate search window. Optional vector compression reduces memory and can accelerate distance calculations. | Graph degree, construction window, search window, and compression. | Approximate search with strong memory/throughput potential, but more build work and platform-dependent compression behavior. |

We use `FLAT` for the executable baseline because it returns exact nearest neighbors across the full WANDS corpus. That removes approximation as a confounding variable while we compare retrieval methods.

For deeper study:

- Documentation for [all three types](https://redis.io/docs/latest/develop/ai/search-and-query/vectors/) and their controls.
- The original [HNSW paper](https://arxiv.org/abs/1603.09320) explains the layered graph.
- Intel's [SVS graph-search guide](https://intel.github.io/ScalableVectorSearch/advanced/graph_search.html) illustrates Vamana traversal and search windows.

> **SVS-VAMANA platform note:** Redis 8.6 supports the index, but Intel's proprietary LVQ and LeanVec optimizations are not available in Redis Open Source and do not run on non-Intel hardware. Redis falls back to basic 8-bit scalar quantization for compressed SVS-VAMANA in those environments. Record the Redis edition and CPU when interpreting any SVS benchmark.

In [ ]:
INDEX_NAME = f'wands_products_{WORKSHOP_RUN_ID}'
INDEX_PREFIX = f'wands:product:{WORKSHOP_RUN_ID}'
LIVE_INDEX_ALGORITHM = 'flat'


def vector_attrs(algorithm=LIVE_INDEX_ALGORITHM):
    attrs = {
        'dims': EMBEDDING_DIMS,
        'distance_metric': 'cosine',
        'algorithm': algorithm,
        'datatype': 'float32',
    }

    if algorithm == 'hnsw':
        attrs.update({'m': 16, 'ef_construction': 200, 'ef_runtime': 20})
    elif algorithm == 'svs-vamana':
        attrs.update({
            'graph_max_degree': 40,
            'construction_window_size': 250,
            'search_window_size': 20,
            'compression': 'LVQ8',
        })

    return attrs

### 3.2 Define the Index Schema

The schema tells Redis Search which fields are searchable as text, tags, numbers, or vectors.

In [ ]:
def make_schema(algorithm=LIVE_INDEX_ALGORITHM):
    return {
        'index': {'name': INDEX_NAME, 'prefix': INDEX_PREFIX, 'storage_type': 'hash'},
        'fields': [
            {'name': 'product_id', 'type': 'tag'},
            {'name': 'product_name', 'type': 'text'},
            {'name': 'product_class', 'type': 'tag'},
            {'name': 'category_hierarchy', 'type': 'text'},
            {'name': 'search_text', 'type': 'text'},
            {'name': 'average_rating', 'type': 'numeric'},
            {'name': 'review_count', 'type': 'numeric'},
            {'name': 'embedding', 'type': 'vector', 'attrs': vector_attrs(algorithm)},
        ],
    }

### 3.3 Create and Load the Index

Each product becomes a Redis hash under the run-scoped `wands:product:<run-id>` prefix. Redis Search indexes matching hashes with the schema above.

The `overwrite=True, drop=True` call is intentional for the workshop: rerunning the notebook recreates the current run's index from the prepared data. Use unique run IDs when multiple people share Redis.

Remote Redis targets can break large write pipelines. The loader uses modest retryable chunks so the same notebook works on local Redis and Redis Cloud.


In [ ]:
from redisvl.index import SearchIndex
import time


def redis_record(row):
    return {
        'product_id': row['product_id'],
        'product_name': row['product_name'],
        'product_class': row['product_class'],
        'category_hierarchy': row['category_hierarchy'],
        'search_text': row['search_text'],
        'average_rating': float(row['average_rating']),
        'review_count': int(row['review_count']),
        'embedding': row['embedding'],
    }


def load_with_retries(index, records, chunk_size, retries=3):
    loaded_keys = []
    total = len(records)

    for start in range(0, total, chunk_size):
        chunk = records[start:start + chunk_size]
        for attempt in range(1, retries + 1):
            try:
                loaded_keys.extend(index.load(chunk, id_field='product_id', batch_size=chunk_size))
                break
            except Exception:
                if attempt == retries:
                    raise
                time.sleep(2 * attempt)

        loaded = min(start + len(chunk), total)
        if loaded == total or loaded % 1000 < chunk_size:
            print(f'Loaded {loaded:,}/{total:,} products into Redis...')

    return loaded_keys

#### Create the Index

Create the live `FLAT` index from the schema. Rerunning this cell recreates only the current run-scoped index.

In [ ]:
index = SearchIndex.from_dict(make_schema(LIVE_INDEX_ALGORITHM), redis_url=REDIS_URL)
print(f'Creating {LIVE_INDEX_ALGORITHM.upper()} index {INDEX_NAME!r} over key prefix {INDEX_PREFIX!r}.')
index.create(overwrite=True, drop=True)

#### Load the Products

Convert all product rows into Redis hashes and load them in retryable batches. Progress output makes the full-corpus load visible without changing which records are processed.

In [ ]:
records = [redis_record(row) for row in corpus_df.to_dict('records')]
loaded_keys = load_with_retries(index, records, chunk_size=REDIS_LOAD_BATCH_SIZE)

print(f"Loaded {len(loaded_keys):,} products into {INDEX_NAME}.")
index.info()

> **Checkpoint:** Product data is stored in Redis hashes, while Redis Search maintains the secondary indexes used by the query patterns that follow.


## 4. Product Search Patterns

Product search is difficult because shoppers and catalogs rarely use identical language. A shopper may type `writing desk`; the catalog may say `study desk`, `desk and chair set`, or `office set`. No single retrieval method handles every case well, so production systems combine complementary signals.

The same patterns support recommendation candidate generation: retrieve a broad set quickly, then apply more expensive reranking or business logic.

> **Live focus:** Compare vector, filtered vector, and hybrid results carefully. The filters section uses one tag constraint and one numeric constraint so participants see both patterns without adding another workshop topic.


In [ ]:
RETURN_FIELDS = [
    'product_id',
    'product_name',
    'product_class',
    'average_rating',
    'vector_distance',
]


def show_results(rows):
    columns = [field for field in RETURN_FIELDS if rows and field in rows[0]]
    return pd.DataFrame(rows)[columns]


demo_vector = vectorizer.embed(demo_query, as_buffer=True, normalize_embeddings=True)

### 4.1 Vector Search

Vector search asks: which products are closest to the shopper query in the embedding space?

In [ ]:
from redisvl.query import VectorQuery


vector_query = VectorQuery(
    vector=demo_vector,
    vector_field_name='embedding',
    return_fields=RETURN_FIELDS,
    num_results=5,
)

print(f"Vector search for: {demo_query!r}")
display(show_results(index.query(vector_query)))

### 4.2 Filtered Vector Search

Filtered vector search answers: what semantic matches remain after a catalog constraint is applied? Redis evaluates the filter as part of the query instead of trimming an already ranked result list.

We will use two common constraint types:

- A **tag filter** for an exact product-class value.
- A **numeric filter** for a minimum rating.

#### Tag Filter: Product Class

A known relevant class stands in for a category selected by a shopper.


In [ ]:
from redisvl.query.filter import Tag

filter_class = teaching_filter_class(demo_qid)
class_filter = Tag('product_class') == filter_class

filtered_vector_query = VectorQuery(
    vector=demo_vector,
    vector_field_name='embedding',
    filter_expression=class_filter,
    return_fields=RETURN_FIELDS,
    num_results=5,
)

print(f"Filtered to product_class={filter_class!r}")
display(show_results(index.query(filtered_vector_query)))

#### Numeric Filter: Minimum Rating

Numeric fields support constraints such as rating, price, inventory, distance, or freshness. Here, semantic ranking runs only over products rated at least 4.0.


In [ ]:
from redisvl.query.filter import Num

rating_filter = Num('average_rating') >= 4
rating_query = VectorQuery(
    vector=demo_vector,
    vector_field_name='embedding',
    filter_expression=rating_filter,
    return_fields=RETURN_FIELDS,
    num_results=5,
)

print('Filtered to average_rating >= 4.0')
display(show_results(index.query(rating_query)))

### 4.4 Hybrid Search

RedisVL's `HybridQuery` sends [`FT.HYBRID`](https://redis.io/docs/latest/commands/ft.hybrid/) commands to Redis, combining a text search leg and a vector similarity leg server-side. Here we use reciprocal rank fusion (`RRF`), which combines rank positions without assuming that text and vector scores share a common scale. Section 6 compares RRF with explicit linear text/vector weights.


In [ ]:
import warnings

from redisvl.query import HybridQuery


with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    hybrid_query = HybridQuery(
        text=demo_query,
        text_field_name='search_text',
        vector=demo_vector,
        vector_field_name='embedding',
        combination_method='RRF',
        yield_combined_score_as='hybrid_score',
        return_fields=['product_id', 'product_name', 'product_class'],
        num_results=5,
    )

    display(pd.DataFrame(index.query(hybrid_query)))

## 5. Benchmarking Vector Indices

Vector index selection and retrieval-method selection are different experiments.

**Section 6** compares different query methods against the one exact `FLAT` index built above. It does **not** tell us whether `HNSW` or `SVS-VAMANA` is the better production index.

A useful index benchmark holds the product records, embedding model, distance metric, query set, and hardware constant. Change only the vector index type and its controls.


### 5.1 Compare at Matched ANN Recall

1. Build separate `FLAT`, `HNSW`, and `SVS-VAMANA` indexes with unique names and key prefixes. Reuse the same embedding bytes so the model is not another variable.
2. Treat `FLAT` top-k neighbors as the exact reference and calculate **ANN Recall@k**: `|approximate top-k intersect exact top-k| / k`.
3. Sweep `EF_RUNTIME` for `HNSW` and `SEARCH_WINDOW_SIZE` for `SVS-VAMANA`. First bring both approximate indexes to a comparable ANN Recall@k target; only then compare latency or throughput. Untuned defaults are not a fair benchmark.
4. At matched ANN Recall@k, measure p50 and p95 latency, queries per second, memory, index build time, and update/ingest behavior. Record corpus size, dimensions, filters, Redis edition, and CPU architecture with the result.
5. Run the WANDS relevance study once per index. ANN Recall@k measures approximation against `FLAT`; nDCG@10 and Recall@25 measure the ranking against human judgments. Keep both because they answer different questions.

## 6. Optimize with WANDS Judgments

Now move to evaluating a repeatable experiment. WANDS relevance judgments let us compare retrieval methods against the same queries and labels. Redis Retrieval Optimizer runs each named method and produces one comparable scorecard with defined metrics.


### 6.1 Define the Scorecard Once

The scorecard uses the same four metric columns for evaluation, display, persistence, and the final recommendation. `ranx` computes each relevance metric per query and reports the mean across all 480 WANDS queries.

| Metric | Exact meaning in this study | Decision role |
|---|---|---|
| nDCG@10 | Normalized discounted cumulative gain over ranks 1-10. WANDS grades are `Exact = 2`, `Partial = 1`, and `Irrelevant = 0`; stronger matches and earlier ranks earn more credit, then each query is normalized by its ideal top ten. Range: 0-1; higher is better. | Primary first-page ranking measure. |
| Recall@25 | For each query, the number of judged-positive products (grade greater than 0) returned in the top 25, divided by all judged-positive products for that query. Range: 0-1; higher is better. Queries with more than 25 positives impose a natural top-25 ceiling. | Candidate-coverage guardrail. |
| Precision@25 | For each query, the number of judged-positive products in the top 25 divided by 25. Range: 0-1; higher is better. Under this offline qrels evaluation, unjudged products receive no relevance credit. | Diagnostic for broad but noisy retrieval. |
| Average Redis query time (`avg_redis_query_ms`) | Mean wall-clock milliseconds around RedisVL `index.query`, including the local Redis round trip and result decoding. Query embedding and Python query construction happen before this timer. Lower is better. | Relative workshop latency signal, not a production load test. |

**Decision rule:** sort by nDCG@10, break relevance ties with Recall@25, and use average Redis query time only as the final tie-breaker. Precision@25 remains visible as a diagnostic rather than a separate sorting objective.

### 6.2 Run One Controlled Study

Every candidate uses the same existing `FLAT` index, all 480 prepared queries, the matching qrels, the same embedding model, and `ret_k = 25`. Only retrieval behavior changes. The notebook reuses `selected_files['queries']` and `selected_files['qrels']`; it does not write a second copy of the evidence.

| Layer | Responsibility |
|---|---|
| This notebook | Fix the experimental inputs, define the scorecard and decision rule once, run the study, and interpret the result. |
| `scripts/workshop_search_study.py` | Adapt RedisVL rows to `ranx.Run`, construct text/vector/hybrid queries, capture runs for cutoff evaluation, and persist the exact displayed scorecard. |
| Redis Retrieval Optimizer | Supply the search-method contract, connect to the existing index, execute every registered method, and collect per-query Redis timings. |

The imported adapter keeps integration mechanics available in the repository without making them the lesson a second time. Section 4 already introduced the query objects; this section teaches how to compare retrieval strategies fairly.

The deliberately small candidate set has six methods:

| Candidate | Question it answers |
|---|---|
| BM25 text | How strong is lexical matching alone? |
| Vector cosine | How strong is semantic matching alone? |
| Hybrid RRF | Does rank fusion improve on either signal? |
| Linear hybrid, text weights 0.25 / 0.50 / 0.75 | Does emphasizing vector, balancing both signals, or emphasizing text work best? |

For linear fusion, `linear_alpha` is the **text** weight. All methods return 25 products, so Recall@25 and Precision@25 compare the same result depth.

In [ ]:
from scripts.workshop_search_study import run_workshop_search_study

STUDY_ID = f'wands-search-study-{WORKSHOP_RUN_ID}'

print(f'Running 6 methods over {len(queries):,} queries...')
study_result = run_workshop_search_study(
    redis_url=REDIS_URL,
    study_id=STUDY_ID,
    index_name=INDEX_NAME,
    queries_path=selected_files['queries'],
    qrels_path=selected_files['qrels'],
    embedding_model=HF_MODEL,
    embedding_dims=EMBEDDING_DIMS,
    embedding_cache_name=f'wands_retopt_embeddings_{WORKSHOP_RUN_ID}',
)

optimizer_df = study_result.scorecard
display(optimizer_df)
print(f'Persisted this exact scorecard at {study_result.redis_key}.')

### 6.3 Make the Recommendation

The first row is the recommendation under the decision rule defined above. Treat near-equal relevance scores and small timing differences from one run as an effective tie. The result applies to WANDS, this index, this embedding model, and this candidate set—not to search systems universally.

**What this experiment still cannot answer**

- WANDS judgments are incomplete. The offline evaluation gives unjudged products no credit even when a user might consider them relevant.
- Three linear weights are an interpretable sweep, not exhaustive tuning; filters, rerankers, and `ret_k` variants remain future candidates.
- The timing is a sequential local Redis measurement and excludes query embedding, concurrency, network distance, and tail latency.
- One offline judged set cannot establish statistical significance, freshness behavior, or changes in online user outcomes.

In production, refresh the judged set, add candidates in `scripts/workshop_search_study.py`, and use repeated or paired significance analysis before treating a small score difference as real. Compare vector index types as separate studies using the matched ANN Recall protocol from Section 5.

**Production checklist**

1. Keep product data fresh with an incremental indexing path, not only full reloads.
2. Use per-environment index names, prefixes, and cache names.
3. Monitor latency, memory, index size, embedding failures, and zero-result queries.
4. Validate the strategy on representative traffic and a larger or refreshed judged set.
5. Roll out with an A/B test or shadow evaluation when user behavior is the final judge.

In [ ]:
best = optimizer_df.iloc[0]

print('Recommendation from this Redis Retrieval Optimizer study')
print(f"- Search method: {best['search_method']}")
print(f"- nDCG@10: {float(best['ndcg@10']):.4f}")
print(f"- Recall@25: {float(best['recall@25']):.4f}")
print(f"- Precision@25: {float(best['precision@25']):.4f}")
print(f"- Average Redis query time: {best['avg_redis_query_ms']} ms")
print(f"- Persisted scorecard: {study_result.redis_key}")

print()
print('Next experiment: add one production-motivated candidate, then rerun the same controlled study.')

## Reference Guide

Use these resources after the live session to go deeper or design the next experiment.

| Topic | Primary resource |
|---|---|
| WANDS data and annotation | [Official WANDS repository](https://github.com/wayfair/WANDS) and [Wayfair's dataset overview](https://www.aboutwayfair.com/careers/tech-blog/wayfair-releases-wands-the-largest-and-richest-publicly-available-dataset-for-e-commerce-product-search-relevance) |
| Workshop embedding model | [`all-MiniLM-L6-v2` model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) |
| Hosted and local embedding providers | [RedisVL vectorizer concepts](https://redis.io/docs/latest/develop/ai/redisvl/concepts/utilities/) |
| Embedding fine-tuning | [Sentence Transformers training overview](https://sbert.net/docs/sentence_transformer/training_overview.html) |
| Redis vector index types and controls | [Redis vector search concepts](https://redis.io/docs/latest/develop/ai/search-and-query/vectors/) |
| HNSW internals | [Original HNSW paper](https://arxiv.org/abs/1603.09320) |
| Vamana graph traversal and tuning | [Intel SVS graph-search guide](https://intel.github.io/ScalableVectorSearch/advanced/graph_search.html) |
| Server-side hybrid retrieval | [`FT.HYBRID` command reference](https://redis.io/docs/latest/commands/ft.hybrid/) |
| Information-retrieval metrics | [ranx metric reference](https://amenra.github.io/ranx/metrics/) |
